# Llama 3.1 8B Instruct: Hexagon Hallucination Benchmark Runner

This uses **zero-shot, fixed instruction templates** (same wording in EN/NE, only the language changes) so results
are directly comparable to the other models  this also fills in the "document exact prompt template" TODO
in Section 6.2 of the paper draft. Decoding is single-pass, temperature=0 (greedy) for reproducibility.

In [1]:
!pip install -q -U transformers accelerate bitsandbytes openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 82.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 67.7 MB/s eta 0:00:00:00:01


In [2]:

!pip install -q -U bitsandbytes

In [3]:
import bitsandbytes; print(bitsandbytes.__version__)


0.50.2


In [4]:
import os, gc, time
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

MODEL_ID = "NousResearch/Meta-Llama-3.1-8B-Instruct"  # ungated mirror, identical weights

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print("Model loaded.")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded.


In [5]:
def generate(prompt, max_new_tokens=200):
    """Single-pass, greedy (temperature=0) generation for reproducibility."""
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[-1]
    text = tokenizer.decode(out[0][input_len:], skip_special_tokens=True)
    return text.strip()

In [6]:
# ---------- Fixed prompt templates (English wording + Nepali wording, same instruction) ----------

QA_PROMPT_EN = (
    "Answer the following question as accurately and concisely as possible. "
    "Give only the factual answer, with no extra commentary.\n\nQuestion: {q}\nAnswer:"
)
QA_PROMPT_NE = (
    "तलको प्रश्नको सकेसम्म सही र छोटो उत्तर दिनुहोस्। कुनै अतिरिक्त टिप्पणी नगरी केवल तथ्यगत उत्तर मात्र दिनुहोस्।"
    "\n\nप्रश्न: {q}\nउत्तर:"
)

FEVER_PROMPT_EN = (
    "Determine whether the following statement is factually True or False. "
    "Respond with exactly one word: True or False.\n\nStatement: {s}\nAnswer:"
)
FEVER_PROMPT_NE = (
    "तलको भनाइ तथ्यगत रूपमा सत्य हो वा असत्य हो निर्धारण गर्नुहोस्। एउटै शब्दमा मात्र जवाफ दिनुहोस्: True वा False।"
    "\n\nभनाइ: {s}\nउत्तर:"
)

SUMMARY_PROMPT_EN = (
    "Summarize the following dialogue in one or two sentences. Only include information "
    "present in the dialogue.\n\nDialogue:\n{d}\n\nSummary:"
)
SUMMARY_PROMPT_NE = (
    "तलको संवादलाई एक वा दुई वाक्यमा सारांश गर्नुहोस्। संवादमा भएको जानकारी मात्र समावेश गर्नुहोस्।"
    "\n\nसंवाद:\n{d}\n\nसारांश:"
)

In [7]:
def qa_prompt(question, language):
    return QA_PROMPT_EN.format(q=question) if language == "English" else QA_PROMPT_NE.format(q=question)

def fever_prompt(statement, language):
    return FEVER_PROMPT_EN.format(s=statement) if language == "English" else FEVER_PROMPT_NE.format(s=statement)

def summary_prompt(dialogue, language):
    return SUMMARY_PROMPT_EN.format(d=dialogue) if language == "English" else SUMMARY_PROMPT_NE.format(d=dialogue)

def parse_true_false(raw):
    t = raw.strip().lower()
    if t.startswith("true"):
        return "True"
    if t.startswith("false"):
        return "False"
    return raw.strip()[:20]  # fallback: keep a short snippet for manual review

In [8]:
TEMPLATE_PATH = "/kaggle/input/datasets/nishashrestha2000/model-evaluation/New_Model_Evaluation_Templates.xlsx"
EXISTING_CONSTITUTION_LAW_PATH = "/kaggle/input/datasets/nishashrestha2000/llama-progress/Constitution_Law_Llama3_1_8B (2).csv"
EXISTING_POLYFEVER_PATH = "/kaggle/input/datasets/nishashrestha2000/llama-progress/Poly_FEVER_Llama3.1_8B (1).csv"
EXISTING_HALLUVERSE_PATH = "/kaggle/input/datasets/nishashrestha2000/llama-progress/HalluVerseM3_Llama3_1_8B.csv"
EXISTING_CULTURE_PATH = "/kaggle/input/datasets/nishashrestha2000/llama-progress/Culture_Religion_Literature_Llama3.1_8B (1).csv"  # <-- add
EXISTING_ECONOMY_PATH = "/kaggle/input/datasets/nishashrestha2000/llama-progress/Economy_Tech_Misc_Llama3.1_8B.csv"  # <-- add
OUT_DIR = "/kaggle/working"
CHECKPOINT_EVERY = 50
MODEL_NAME = "Llama3.1_8B"

sheets = pd.read_excel(TEMPLATE_PATH, sheet_name=None)
print(list(sheets.keys()))

['README', 'Education_Health', 'Geography_Tourism', 'Constitution_Law', 'Culture_Religion_Literature', 'Economy_Tech_Misc', 'Poly_FEVER', 'HalluVerseM3']


In [9]:
import pandas as pd

for label, path in [
    ("Constitution_Law", EXISTING_CONSTITUTION_LAW_PATH),
    ("Poly_FEVER", EXISTING_POLYFEVER_PATH),
    ("HalluVerseM3", EXISTING_HALLUVERSE_PATH),
    ("Culture_Religion_Literature", EXISTING_CULTURE_PATH),  # <-- add
    ("Economy_Tech_Misc", EXISTING_ECONOMY_PATH),  # <-- add
]:
    df = pd.read_csv(path)
    ans_col = [c for c in df.columns if "Answer" in c or "Summary" in c][-1]
    n_ok = df[ans_col].notna().sum() - df[ans_col].astype(str).str.startswith("[ERROR").sum()
    print(f"{label}: {len(df)} rows, {n_ok} answered")

Constitution_Law: 1826 rows, 1826 answered
Poly_FEVER: 12002 rows, 12002 answered
HalluVerseM3: 398 rows, 301 answered
Culture_Religion_Literature: 848 rows, 848 answered
Economy_Tech_Misc: 266 rows, 266 answered


In [10]:
import shutil, os

seed_files = {
    "Constitution_Law": EXISTING_CONSTITUTION_LAW_PATH,
    "Poly_FEVER": EXISTING_POLYFEVER_PATH,
    "HalluVerseM3": EXISTING_HALLUVERSE_PATH,
    "Culture_Religion_Literature": EXISTING_CULTURE_PATH,  # <-- add
    "Economy_Tech_Misc": EXISTING_ECONOMY_PATH,  # <-- add
}

for sheet_name, src_path in seed_files.items():
    dest = f"{OUT_DIR}/{sheet_name}_{MODEL_NAME}.csv"
    if os.path.exists(src_path):
        shutil.copy(src_path, dest)
        print(f"Seeded {sheet_name} checkpoint with existing progress.")
    else:
        print(f"WARNING: existing file not found for {sheet_name} at", src_path)

Seeded Constitution_Law checkpoint with existing progress.
Seeded Poly_FEVER checkpoint with existing progress.
Seeded HalluVerseM3 checkpoint with existing progress.
Seeded Culture_Religion_Literature checkpoint with existing progress.
Seeded Economy_Tech_Misc checkpoint with existing progress.


In [11]:
def run_domain_sheet(name, df):
    """Fills [MODEL] Answer for one of the 5 domain QA sheets."""
    out_path = f"{OUT_DIR}/{name}_{MODEL_NAME}.csv"
    if os.path.exists(out_path):
        df = pd.read_csv(out_path)
    answer_col = "[MODEL] Answer"
    if answer_col not in df.columns:
        df[answer_col] = ""
    df[answer_col] = df[answer_col].fillna("")
    for i, row in df.iterrows():
        if str(row[answer_col]).strip():
            continue
        prompt = qa_prompt(row["Question"], row["Language"])
        try:
            df.at[i, answer_col] = generate(prompt, max_new_tokens=150)
        except Exception as e:
            df.at[i, answer_col] = f"[ERROR: {e}]"
        if i % CHECKPOINT_EVERY == 0:
            df.to_csv(out_path, index=False)
            print(f"{name}: checkpoint at row {i}/{len(df)}")
    df.to_csv(out_path, index=False)
    print(f"{name}: done, saved to {out_path}")
    return df

domain_sheets = ["Constitution_Law", "Culture_Religion_Literature", "Economy_Tech_Misc"]  # Constitution_Law will skip instantly (already seeded complete)

results = {}
for name in domain_sheets:
    results[name] = run_domain_sheet(name, sheets[name].copy())

Constitution_Law: done, saved to /kaggle/working/Constitution_Law_Llama3.1_8B.csv
Culture_Religion_Literature: done, saved to /kaggle/working/Culture_Religion_Literature_Llama3.1_8B.csv
Economy_Tech_Misc: done, saved to /kaggle/working/Economy_Tech_Misc_Llama3.1_8B.csv


In [12]:
import os, pandas as pd
out_path = f"{OUT_DIR}/Poly_FEVER_{MODEL_NAME}.csv"
print("exists:", os.path.exists(out_path))
if os.path.exists(out_path):
    df = pd.read_csv(out_path)
    print(len(df), "rows,", df["[MODEL] Answer"].notna().sum(), "answered")

exists: True
12002 rows, 12002 answered


In [13]:
!nvidia-smi

Thu Sep  3 07:46:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P0             28W /   70W |    1691MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [14]:
import time
t0 = time.time()
print(generate("Hello, how are you?", max_new_tokens=5))
print("took", time.time() - t0, "seconds")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


I'm just a language
took 1.9657206535339355 seconds


In [15]:
def run_polyfever(df):
    out_path = f"{OUT_DIR}/Poly_FEVER_{MODEL_NAME}.csv"
    if os.path.exists(out_path):
        df = pd.read_csv(out_path)
    raw_col, ans_col = "[MODEL] Raw Output", "[MODEL] Answer"
    for c in (raw_col, ans_col):
        if c not in df.columns:
            df[c] = ""
        df[c] = df[c].fillna("")
    for i, row in df.iterrows():
        if str(row[ans_col]).strip():
            continue
        prompt = fever_prompt(row["Statement"], row["Language"])
        try:
            raw = generate(prompt, max_new_tokens=10)
            df.at[i, raw_col] = raw
            df.at[i, ans_col] = parse_true_false(raw)
        except Exception as e:
            df.at[i, raw_col] = f"[ERROR: {e}]"
        if i % CHECKPOINT_EVERY == 0:
            df.to_csv(out_path, index=False)
            print(f"Poly_FEVER: checkpoint at row {i}/{len(df)}")
    df.to_csv(out_path, index=False)
    print(f"Poly_FEVER: done, saved to {out_path}")
    return df

results["Poly_FEVER"] = run_polyfever(sheets["Poly_FEVER"].copy())

Poly_FEVER: done, saved to /kaggle/working/Poly_FEVER_Llama3.1_8B.csv


In [16]:
def run_halluverse(df):
    out_path = f"{OUT_DIR}/HalluVerseM3_{MODEL_NAME}.csv"
    if os.path.exists(out_path):
        df = pd.read_csv(out_path)
    raw_col, sum_col = "[MODEL] Raw Output", "[MODEL] Summary"
    for c in (raw_col, sum_col):
        if c not in df.columns:
            df[c] = ""
        df[c] = df[c].fillna("")
    for i, row in df.iterrows():
        if str(row[sum_col]).strip():
            continue
        prompt = summary_prompt(row["Dialogue"], row["Language"])
        try:
            raw = generate(prompt, max_new_tokens=150)
            df.at[i, raw_col] = raw
            df.at[i, sum_col] = raw
        except Exception as e:
            df.at[i, raw_col] = f"[ERROR: {e}]"
        if i % CHECKPOINT_EVERY == 0:
            df.to_csv(out_path, index=False)
            print(f"HalluVerseM3: checkpoint at row {i}/{len(df)}")
    df.to_csv(out_path, index=False)
    print(f"HalluVerseM3: done, saved to {out_path}")
    return df

results["HalluVerseM3"] = run_halluverse(sheets["HalluVerseM3"].copy())

HalluVerseM3: checkpoint at row 350/398
HalluVerseM3: done, saved to /kaggle/working/HalluVerseM3_Llama3.1_8B.csv


In [17]:
# ---------- Combine everything back into one workbook, matching the template layout ----------
final_path = f"{OUT_DIR}/{MODEL_NAME}_All_Results.xlsx"
with pd.ExcelWriter(final_path, engine="openpyxl") as writer:
    for name in domain_sheets:
        results[name].to_excel(writer, sheet_name=name[:31], index=False)
    results["Poly_FEVER"].to_excel(writer, sheet_name="Poly_FEVER", index=False)
    results["HalluVerseM3"].to_excel(writer, sheet_name="HalluVerseM3", index=False)

print("Saved:", final_path)
print("Download it from the Kaggle notebook's Output/Data pane.")

Saved: /kaggle/working/Llama3.1_8B_All_Results.xlsx
Download it from the Kaggle notebook's Output/Data pane.


In [18]:
import pandas as pd
xl = pd.read_excel("/kaggle/working/Llama3.1_8B_All_Results.xlsx", sheet_name=None)

for name, df in xl.items():
    if name == "README":
        continue
    ans_col = [c for c in df.columns if "Answer" in c or "Summary" in c]
    if ans_col:
        col = ans_col[-1]
        n_error = df[col].astype(str).str.startswith("[ERROR").sum()
        n_empty = (df[col].astype(str).str.strip() == "").sum()
        print(f"{name}: {len(df)} rows, {n_error} errors, {n_empty} empty")
        print("  sample:", df[col].dropna().astype(str).head(3).tolist())

Constitution_Law: 1826 rows, 0 errors, 0 empty
  sample: ['The Constitution of Nepal 2015 is the fundamental law of Nepal.', 'नेपालको मौलिक कानून नेपालको संविधान हो ।', 'The people of Nepal.']
Culture_Religion_Literature: 848 rows, 0 errors, 0 empty
  sample: ['15 days', 'दशैं पर्व १० दिनसम्म मनाइन्छ ।', 'Kukur Tihar']
Economy_Tech_Misc: 266 rows, 0 errors, 0 empty
  sample: ['Nepalese rupee (NPR)', 'नेपालको आधिकारिक मुद्रा नेपालको रुपैया हो ।', 'NPR']
Poly_FEVER: 12002 rows, 0 errors, 0 empty
  sample: ['False', 'True', 'True']
HalluVerseM3: 398 rows, 0 errors, 0 empty
  sample: ['A customer wants to buy a toy car for her son and is shown two options, one for $300 and another for $120. She chooses the cheaper option and pays for it.', 'संवादमा भएको जानकारी मात्र समावेश गर्दै संवादलाई एक वाक्यमा सारांश गर्दै यस प्रकार लेख्न सकिन्छ: महोदया म तपाईंका लागि खेलौना गाडी किन्न चाहन्छु।', "Steve and an unnamed person run into each other and catch up, discussing the person's recent start to wo

## Notes

- **Checkpointing**: each sheet writes a `.csv` checkpoint to `/kaggle/working/` every 50 rows, and skips
  rows that already have an answer so if the Kaggle session times out (9 hr GPU limit) you can just
  re-run the notebook and it will resume where it left off, as long as `/kaggle/working` persists (Save Version).

  
- **Greedy decoding** (`do_sample=False`) is used for reproducibility, matching a fixed zero-shot (temperature=0, single-pass, same instruction template in both
  languages).

In [ ]:
import traceback

# Grab one real question from your data to test with
test_q = sheets["Education_Health"].iloc[0]["Question"]
test_lang = sheets["Education_Health"].iloc[0]["Language"]
print("Testing with:", test_q, "|", test_lang)

try:
    result = generate(qa_prompt(test_q, test_lang), max_new_tokens=50)
    print("SUCCESS:", result)
except Exception:
    traceback.print_exc()   # full traceback, not just str(e)

In [ ]:
import shutil

shutil.make_archive('/kaggle/working/all_results', 'zip', '/kaggle/working')
print("Zipped everything into /kaggle/working/all_results.zip")

In [ ]:
import os
for f in sorted(os.listdir('/kaggle/working')):
    path = os.path.join('/kaggle/working', f)
    if os.path.isfile(path):
        print(f, "-", round(os.path.getsize(path)/1024/1024, 2), "MB")
    else:
        print(f, "(folder)")

In [ ]:
from IPython.display import FileLink, Javascript, display
import os, time

def auto_download(path):
    display(FileLink(path))
    display(Javascript("""
        var links = document.querySelectorAll('a');
        var target = links[links.length - 1];
        if (target) { target.click(); }
    """))
    time.sleep(1.5)  # give the browser a moment before triggering the next one

files_to_grab = [
    "/kaggle/working/Education_Health_Llama3.1_8B.csv",
    "/kaggle/working/Geography_Tourism_Llama3.1_8B.csv",
    "/kaggle/working/Constitution_Law_Llama3.1_8B.csv",
    "/kaggle/working/Culture_Religion_Literature_Llama3.1_8B.csv",
    "/kaggle/working/Economy_Tech_Misc_Llama3.1_8B.csv",
    "/kaggle/working/HalluVerseM3_Llama3.1_8B.csv",
]

for f in files_to_grab:
    if os.path.exists(f):
        print("Downloading:", f)
        auto_download(f)
    else:
        print("MISSING (didn't finish?):", f)